In [1]:
import cvxpy as cp
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
iris = sns.load_dataset('iris')
print("Initial Dataset Preview:")
print(iris.head())

Initial Dataset Preview:
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa


In [3]:
species_map = {'setosa': 1,'versicolor': 2,'virginica': 3}
iris['target'] = iris['species'].map(species_map)

# Feature matrix X (4 features) and target vector y
feature_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris[feature_cols].values
y = iris['target'].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")
print("Train class counts:", pd.Series(y_train).value_counts().to_dict())
print("Test class counts:", pd.Series(y_test).value_counts().to_dict())


Training set size: 75, Test set size: 75
Train class counts: {1: 25, 2: 25, 3: 25}
Test class counts: {1: 25, 3: 25, 2: 25}


In [10]:
n_train, d = X_train.shape

w = cp.Variable(d)
b = cp.Variable()

# Objective: minimize mean squared error on training split
objective = cp.Minimize((1.0 / n_train) * cp.sum_squares(X_train @ w + b - y_train))
problem = cp.Problem(objective)
problem.solve()

print("\n--- Model Optimization Results ---")
print(f"Solver Status: {problem.status}")
print(f"Optimal weights (w):\n{w.value}")
print(f"Optimal bias (b): {b.value:.4f}")
print(f"Optimal Training Loss: {problem.value:.4f}")


--- Model Optimization Results ---
Solver Status: optimal
Optimal weights (w):
[-0.14252541  0.01162351  0.25071717  0.5897505 ]
Optimal bias (b): 1.1404
Optimal Training Loss: 0.0389


In [12]:
y_test_pred = X_test @ w.value + b.value

# Create a comparison table for inspection
results_df = pd.DataFrame({
    'True Label': y_test,
    'Predicted Value': np.round(y_test_pred, 4)
})

print("\n--- Predictions on Test Split ---")
print(results_df.head(15))


--- Predictions on Test Split ---
    True Label  Predicted Value
0            1           1.0685
1            1           0.9614
2            1           0.8838
3            1           1.1730
4            3           2.9623
5            1           0.9714
6            2           2.1635
7            3           2.7829
8            2           1.8871
9            3           2.7682
10           2           1.9979
11           1           0.9944
12           2           2.2341
13           2           2.2783
14           1           1.0269
